# 🤖 Soumyadipta Seal — Portfolio Chatbot (ML Intent Classifier)

This notebook trains a lightweight **Machine Learning intent-classification chatbot** that can answer questions about my:
- Skills
- Work experience / internships
- Projects
- Education
- Achievements
- Certifications
- Contact info (GitHub, LinkedIn, Email, Phone)

**Approach:** TF-IDF vectorization + a Linear SVM (via `SGDClassifier`, calibrated for probabilities) trained on `portfolio_chatbot_data.csv` (text → intent). Each predicted intent is mapped to a hand-written response built from my resume.

At the end, the trained **model + vectorizer + response map** are exported as `.pkl` / `.json` files that `app.py` loads to serve the chatbot on Render.


In [1]:
import pandas as pd
import numpy as np
import re
import string
import pickle
import json

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

print("Libraries loaded successfully ✅")


Libraries loaded successfully ✅


## 1. Load the training data

In [2]:
df = pd.read_csv("portfolio_chatbot_data.csv")
print("Shape:", df.shape)
df.head(10)


Shape: (237, 2)


,text,intent
0,hi,greeting
1,hello,greeting
2,hey there,greeting
3,good morning,greeting
4,good evening,greeting
5,hey,greeting
6,is anyone there,greeting
7,hello bot,greeting
8,hi there,greeting
9,yo,greeting


In [3]:
print("Number of intents:", df['intent'].nunique())
df['intent'].value_counts()


Number of intents: 15


intent
skills             31
projects           30
experience         24
achievements       18
greeting           17
about              17
contact            16
education          15
certifications     11
github             11
goodbye            11
linkedin            9
thanks              9
profession          9
extracurricular     9
Name: count, dtype: int64

## 2. Text preprocessing

We lowercase, strip punctuation, and normalize whitespace before vectorizing. Keeping it simple works well for a small, well-structured FAQ-style dataset like this.


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[%s]" % re.escape(string.punctuation), " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text', 'intent']].head()


,text,clean_text,intent
0,hi,hi,greeting
1,hello,hello,greeting
2,hey there,hey there,greeting
3,good morning,good morning,greeting
4,good evening,good evening,greeting


## 3. Train / test split

Our dataset is intentionally small (a curated FAQ-style corpus, not a huge web-scraped one), so a single 80/20 split
would leave only 1-2 examples per class in the test set — too noisy to trust. Instead we:

1. Hold out a stratified test split **just to sanity-check generalization**.
2. Separately, use **5-fold stratified cross-validation** over the *entire* dataset to get a more reliable accuracy estimate.
3. Train the **final production model on 100% of the data** (step 7) — for a small, closed-domain FAQ bot, using every
   labeled example we have gives the best real-world coverage, since there's no "unseen future data" beyond the intents we've defined.


In [5]:
X = df['clean_text']
y = df['intent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))


Train size: 189
Test size: 48


## 4. TF-IDF Vectorization

In [6]:
vectorizer_eval = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)

X_train_tfidf = vectorizer_eval.fit_transform(X_train)
X_test_tfidf = vectorizer_eval.transform(X_test)

print("Vocabulary size:", len(vectorizer_eval.vocabulary_))
print("Train matrix shape:", X_train_tfidf.shape)


Vocabulary size: 697
Train matrix shape: (189, 697)


## 5. Train & evaluate the ML model

We use **Logistic Regression** on top of TF-IDF features — a strong, well-calibrated baseline for short-text intent
classification, especially with limited data (it tends to generalize better than margin-based models like SVM when
classes have very few examples). It also natively outputs class probabilities, which we use for a confidence-based fallback.


In [7]:
eval_model = LogisticRegression(
    C=5.0,
    max_iter=1000,
    class_weight='balanced'
)
eval_model.fit(X_train_tfidf, y_train)

y_pred = eval_model.predict(X_test_tfidf)
acc = accuracy_score(y_test, y_pred)
print(f"Held-out Test Accuracy: {acc*100:.2f}%\n")
print(classification_report(y_test, y_pred, zero_division=0))


Held-out Test Accuracy: 56.25%

                 precision    recall  f1-score   support

          about       0.67      0.50      0.57         4
   achievements       0.33      0.25      0.29         4
 certifications       0.00      0.00      0.00         2
        contact       0.33      0.33      0.33         3
      education       0.00      0.00      0.00         3
     experience       0.67      0.80      0.73         5
extracurricular       0.00      0.00      0.00         2
         github       0.67      1.00      0.80         2
        goodbye       0.00      0.00      0.00         2
       greeting       0.75      1.00      0.86         3
       linkedin       1.00      1.00      1.00         2
     profession       0.25      0.50      0.33         2
       projects       0.67      0.67      0.67         6
         skills       0.83      0.83      0.83         6
         thanks       1.00      1.00      1.00         2

       accuracy                           0.56        

### 5-fold cross-validation (more reliable estimate on this small dataset)

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

full_tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
X_full_tfidf = full_tfidf_vectorizer.fit_transform(X)

cv_model = LogisticRegression(C=5.0, max_iter=1000, class_weight='balanced')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(cv_model, X_full_tfidf, y, cv=skf)

print("Cross-validation accuracy per fold:", np.round(cv_scores, 3))
print(f"Mean CV Accuracy: {cv_scores.mean()*100:.2f}% (+/- {cv_scores.std()*100:.2f}%)")


Cross-validation accuracy per fold: [0.625 0.688 0.681 0.681 0.723]
Mean CV Accuracy: 67.95% (+/- 3.15%)


In [9]:
# Confusion matrix on the held-out split, for a quick visual sanity check
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df


,about,achievements,certifications,contact,education,experience,extracurricular,github,goodbye,greeting,linkedin,profession,projects,skills,thanks
about,2,0,0,1,0,0,0,0,0,0,0,1,0,0,0
achievements,0,1,2,0,0,0,0,0,0,0,0,0,1,0,0
certifications,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0
contact,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0
education,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0
experience,0,1,0,0,0,4,0,0,0,0,0,0,0,0,0
extracurricular,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0
github,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0
goodbye,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0
greeting,0,0,0,0,0,0,0,0,0,3,0,0,0,0,0


## 6. Train the FINAL model on 100% of the data

For deployment, we refit the TF-IDF vectorizer + Logistic Regression on **every** labeled example (not just the 80%
training split). This is standard practice for small, closed-domain chatbots — we've already validated the approach
above with cross-validation, so now we maximize the data the production model actually gets to learn from.


In [10]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=1,
    sublinear_tf=True
)
X_all_tfidf = vectorizer.fit_transform(X)

base_model = LogisticRegression(C=5.0, max_iter=1000, class_weight='balanced')
model = CalibratedClassifierCV(base_model, cv=5)
model.fit(X_all_tfidf, y)

print("Final production model trained on", X_all_tfidf.shape[0], "examples ✅")


Final production model trained on 237 examples ✅


## 7. Build a prediction helper with confidence threshold

In [11]:
CONFIDENCE_THRESHOLD = 0.28  # tune as needed

def predict_intent(text, threshold=CONFIDENCE_THRESHOLD):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    probs = model.predict_proba(vec)[0]
    classes = model.classes_
    best_idx = np.argmax(probs)
    best_intent = classes[best_idx]
    best_prob = probs[best_idx]

    if best_prob < threshold:
        return "fallback", best_prob
    return best_intent, best_prob

# quick sanity checks
for q in ["what is your github", "tell me about your internships", "asdkj random gibberish"]:
    intent, prob = predict_intent(q)
    print(f"{q!r:45s} -> intent={intent:15s} confidence={prob:.2f}")


'what is your github'                         -> intent=github          confidence=0.57
'tell me about your internships'              -> intent=experience      confidence=0.49
'asdkj random gibberish'                      -> intent=fallback        confidence=0.24


## 8. Map each intent → a response built from my resume

These responses are hand-written using details from my CV (Soumyadipta Seal — Computer Science Engineering, Narula Institute of Technology).


In [12]:
responses = {
    "greeting": "Hey! 👋 I'm Soumyadipta's portfolio chatbot. Ask me about my skills, projects, work experience, education, or how to reach me!",

    "about": "I'm Soumyadipta Seal, a 3rd-year Computer Science Engineering student at Narula Institute of Technology (CGPA 8.9). "
             "I have experience across three internships in full-stack web development, AI/ML engineering and Android development, "
             "with one published patent, and I was a Top 50 finalist in a state-level hackathon (500+ teams). "
             "I'm proficient in Java, C++, Python, and JavaScript, and skilled in full-stack development, OOP, and database management.",

    "profession": "I'm a Computer Science Engineering undergraduate (3rd year, B.Tech CSE, 2023–2027) at Narula Institute of Technology, "
                  "focused on full-stack development and AI/ML.",

    "skills": "My technical skills include:\n"
              "• Languages: Java, C, C++, Python, JavaScript, SQL\n"
              "• Web Dev: React.js, Node.js, Express.js, MERN, Django\n"
              "• AI/ML: Gemini API, Scikit-learn, Feature Engineering\n"
              "• Mobile: Android Development (Java)\n"
              "• Cloud/Tools: Git, GitHub, Postman, Salesforce, REST APIs\n"
              "• Databases: MongoDB, SQL\n"
              "• CS Core: DSA, OS, DBMS, OOP, AI",

    "experience": "I've completed three internships:\n"
                  "• Software Development & AI Intern — Euphoria GenX TechPark, Kolkata (Jan–Mar 2026): Built an end-to-end "
                  "Diabetes Risk Prediction system using Flask, React.js and an SVC ML model, with RESTful APIs connecting the model to the frontend.\n"
                  "• Full Stack MERN Web Development Intern — Prodigy Infotech (Aug–Sep 2025): Built full-stack web apps using the "
                  "MERN stack and Next.js, designed REST APIs, and optimized MongoDB schemas in an Agile workflow.\n"
                  "• AI/ML Intern — Eduskill, Kolkata (Jul–Sep 2025): Trained ML models with full preprocessing/feature engineering, "
                  "and built an AI pipeline using the Google Gemini API for text generation and classification.",

    "projects": "Some of my key projects:\n"
                "• AudioVisualEmotion — a real-time emotion detection system combining facial expressions (OpenCV) and audio analysis.\n"
                "• Heart Disease Prediction — ML model (Logistic Regression, Decision Trees, SVM) presented as a research paper at IEEE CIACON 2025.\n"
                "• Real-Time Chat App with AI Assistant (MERN) — WebSockets-based chat with an AI assistant, auth, and scalable APIs. "
                "Live: https://chat-app-x2ht.onrender.com/\n"
                "• Auron's Blog Platform (MERN) — a feature-rich blogging platform with auth, content creation, and role-based access control.\n"
                "• MedCare — a real-time doctor appointment booking platform.\n"
                "• Lpglot — an AI-powered smart LPG safety monitoring system.\n"
                "• Krishi Predict — a real-time AI crop yield prediction app for farmers.\n"
                "• Real-Time AI Weather Forecast App — live weather and air-quality insights.\n"
                "Check out the live demos and source code on my GitHub!",

    "education": "I'm pursuing a B.Tech in Computer Science Engineering at Narula Institute of Technology (2023–2027), CGPA 8.9. "
                 "I completed my Class XII (CBSE, 71.2%) and Class X (CBSE, 83.0%) at Barasat Indira Gandhi Memorial High School.",

    "achievements": "A few highlights:\n"
                    "• Published a Patent on a Diabetes Risk Prediction System\n"
                    "• Ranked 44th / Top 50 in Smart India Hackathon 2025 (150+ teams) — Project ZenFit\n"
                    "• Finalist at AI Utkarsh Summit 2026 HACK-O-NiT (250+ teams)\n"
                    "• Co-authored & presented a research paper at IEEE CIACON 2025\n"
                    "• Completed a 100-day LeetCode DSA challenge streak\n"
                    "• Represented my college at IIT Kharagpur's Kshitij national technical fest",

    "certifications": "My certifications include: Generative AI (30h) - Ardent (2026), AI-ML Internship - EduSkills & Google (2025), "
                       "Java - NPTEL (Elite, 92%), Python for Data Science - IBM, Postman API Expert, HackerRank (Java/Python/Problem Solving), "
                       "and Object-Oriented Programming - Udemy.",

    "extracurricular": "Outside of academics, I've been a Hackathon Team Leader, kept a 100-day LeetCode streak, represented my college "
                        "at IIT Kharagpur's Kshitij fest, and taken part in several college programming contests.",

    "contact": "You can reach me at:\n"
               "📧 Email: s.seal.a.b.c@gmail.com\n"
               "📞 Phone: +91 7687967008\n"
               "📍 Location: Kolkata, West Bengal, India\n"
               "💻 GitHub: https://github.com/sseal2004\n"
               "🔗 LinkedIn: linkedin.com/in/soumyadiptaseal-a6633a290",

    "github": "Here's my GitHub profile: https://github.com/sseal2004 — you'll find source code for my projects there, including the "
              "chat app, blog platform, and more.",

    "linkedin": "You can connect with me on LinkedIn: linkedin.com/in/soumyadiptaseal-a6633a290",

    "thanks": "You're welcome! 😊 Let me know if you'd like to know more about my skills, projects, or experience.",

    "goodbye": "Thanks for stopping by! Feel free to reach out at s.seal.a.b.c@gmail.com or connect on GitHub/LinkedIn. Have a great day! 👋",

    "fallback": "Hmm, I'm not sure about that one 🤔 — try asking me about my skills, projects, work experience, education, "
                "achievements, or how to contact me!"
}

print("Total responses defined:", len(responses))


Total responses defined: 16


In [13]:
def chatbot_reply(user_text):
    intent, prob = predict_intent(user_text)
    return responses.get(intent, responses["fallback"]), intent, prob

# Try it out
test_questions = [
    "hi there",
    "what are your skills",
    "tell me about your internship at euphoria genx",
    "what projects have you built",
    "how can I contact you",
    "what's your github",
    "asdlkjasdkj nonsense text"
]

for q in test_questions:
    reply, intent, prob = chatbot_reply(q)
    print(f"You: {q}")
    print(f"Bot [{intent}, conf={prob:.2f}]: {reply}\n")


You: hi there
Bot [greeting, conf=0.70]: Hey! 👋 I'm Soumyadipta's portfolio chatbot. Ask me about my skills, projects, work experience, education, or how to reach me!

You: what are your skills
Bot [skills, conf=0.56]: My technical skills include:
• Languages: Java, C, C++, Python, JavaScript, SQL
• Web Dev: React.js, Node.js, Express.js, MERN, Django
• AI/ML: Gemini API, Scikit-learn, Feature Engineering
• Mobile: Android Development (Java)
• Cloud/Tools: Git, GitHub, Postman, Salesforce, REST APIs
• Databases: MongoDB, SQL
• CS Core: DSA, OS, DBMS, OOP, AI

You: tell me about your internship at euphoria genx
Bot [experience, conf=0.64]: I've completed three internships:
• Software Development & AI Intern — Euphoria GenX TechPark, Kolkata (Jan–Mar 2026): Built an end-to-end Diabetes Risk Prediction system using Flask, React.js and an SVC ML model, with RESTful APIs connecting the model to the frontend.
• Full Stack MERN Web Development Intern — Prodigy Infotech (Aug–Sep 2025): Built f

## 9. Export the model artifacts

We save:
- `model.pkl` — the trained calibrated SVM classifier
- `vectorizer.pkl` — the fitted TF-IDF vectorizer
- `responses.json` — the intent → response mapping

These three files are loaded by `app.py` to serve the chatbot via a Flask API (ready to deploy on Render).


In [14]:
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

with open("responses.json", "w") as f:
    json.dump(responses, f, indent=2)

print("Saved: model.pkl, vectorizer.pkl, responses.json ✅")


Saved: model.pkl, vectorizer.pkl, responses.json ✅


## 10. Next steps

Copy `model.pkl`, `vectorizer.pkl`, `responses.json`, and `app.py` into the same folder, then deploy on **Render** as a Python web service:

1. Push this folder to a GitHub repo (e.g. `https://github.com/sseal2004/portfolio-chatbot`).
2. On [render.com](https://render.com) → **New +** → **Web Service** → connect the repo.
3. Build command: `pip install -r requirements.txt`
4. Start command: `gunicorn app:app`
5. Deploy — Render will give you a live URL you can call from your portfolio website's frontend.

See `app.py` for the Flask API (`/chat` POST endpoint) and a minimal built-in chat UI at `/`.
